[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 协同过滤
-----

在这个例子里，我们将构建一个快速的显式反馈推荐系统：也就是说，一个考虑显式反馈信号（比如评分）来推荐新内容的模型。

我们将使用一个由 [Netflix 大奖](https://en.wikipedia.org/wiki/Netflix_Prize) 竞赛推广开来的方法：[矩阵分解](https://datajobs.com/data-science-repo/Recommender-Systems-[Netflix].pdf)。

基本思想非常简单：

1. 从用户-物品-评分三元组出发，它表达的信息是：用户 _i_ 给物品 _j_ 打了 _r_ 分。
2. 把用户和物品都表示成高维数值向量。例如，一个用户可以用 `[0.3, -1.2, 0.5]` 表示，一个物品用 `[1.0, -0.3, -0.6]` 表示。
3. 表示应该选得让（通过[点积](https://en.wikipedia.org/wiki/Dot_product)）相乘后能恢复出原始评分。
4. 模型的用途就在于：如果用某个用户的用户向量去乘一个他们_没有_评过分的物品的物品向量，我们希望得到他们如果看过这个物品会给出的评分预测。

![协同过滤](matrix_factorization.png)


## 1. 准备工作


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import os.path as op

from zipfile import ZipFile
try:
    from urllib.request import urlretrieve
except ImportError:  # except ImportError:  # Python 2 兼容
    from urllib import urlretrieve

# 如果不是在 colab 上，这一行需要修改
data_folder = '/content/'

ML_100K_URL = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"
ML_100K_FILENAME = op.join(data_folder,ML_100K_URL.rsplit('/', 1)[1])
ML_100K_FOLDER = op.join(data_folder,'ml-100k')

我们从一个著名的数据集开始，[Movielens 100k 数据集](https://grouplens.org/datasets/movielens/100k/)。它包含 943 个用户对 1682 部电影的 100,000 条评分（1 到 5 分）：


In [ ]:
if not op.exists(ML_100K_FILENAME):
    print('Downloading %s to %s...' % (ML_100K_URL, ML_100K_FILENAME))
    urlretrieve(ML_100K_URL, ML_100K_FILENAME)

if not op.exists(ML_100K_FOLDER):
    print('Extracting %s to %s...' % (ML_100K_FILENAME, ML_100K_FOLDER))
    ZipFile(ML_100K_FILENAME).extractall(data_folder)

其他数据集，见：[Movielens](https://grouplens.org/datasets/movielens/)


## 2. 数据分析与格式化


[Python 数据分析库](http://pandas.pydata.org/)


In [ ]:
import pandas as pd

all_ratings = pd.read_csv(op.join(ML_100K_FOLDER, 'u.data'), sep='\t',
                          names=["user_id", "item_id", "ratings", "timestamp"])
all_ratings.head()

让我们看看数据集的一些宏观统计


In [ ]:
list_movies_names = []
list_item_ids = []
with open(op.join(ML_100K_FOLDER, 'u.item'), encoding = "ISO-8859-1") as fp:
    for line in fp:
        list_item_ids.append(line.split('|')[0])
        list_movies_names.append(line.split('|')[1])
        
movies_names = pd.DataFrame(list(zip(list_item_ids, list_movies_names)), 
               columns =['item_id', 'item_name']) 
movies_names.head()

In [ ]:
movies_names['item_id']=movies_names['item_id'].astype(int)
all_ratings['item_id']=all_ratings['item_id'].astype(int)

In [ ]:
all_ratings = all_ratings.merge(movies_names,on='item_id')
all_ratings.head()

In [ ]:
#条目数量
len(all_ratings)

In [ ]:
all_ratings['ratings'].describe()

In [ ]:
# 不同评分值的数量
len(all_ratings['ratings'].unique())

In [ ]:
all_ratings['user_id'].describe()

In [ ]:
# 不同用户的数量
total_user_id = len(all_ratings['user_id'].unique())
print(total_user_id)

In [ ]:
all_ratings['item_id'].describe()

In [ ]:
# 被评过分的不同物品数量
total_item_id = len(all_ratings['item_id'].unique())
print(total_item_id)

In [ ]:
all_ratings['item_id'] = all_ratings['item_id'].apply(lambda x :x-1)
all_ratings['user_id'] = all_ratings['user_id'].apply(lambda x :x-1)

In [ ]:
movies_names['item_id']=movies_names['item_id'].apply(lambda x: x-1)

In [ ]:
movies_names=movies_names.set_index('item_id')

In [ ]:
movies_names.head()

为了把数据分成 _train_ 和 _test_，我们将使用 [scikit-learn](http://scikit-learn.org/stable/) 里一个现成的函数


In [ ]:
from sklearn.model_selection import train_test_split

ratings_train, ratings_test = train_test_split(
    all_ratings, test_size=0.2, random_state=42)

user_id_train = ratings_train['user_id']
item_id_train = ratings_train['item_id']
rating_train = ratings_train['ratings']

user_id_test = ratings_test['user_id']
item_id_test = ratings_test['item_id']
rating_test = ratings_test['ratings']

In [ ]:
len(user_id_train)

In [ ]:
len(user_id_train.unique())

In [ ]:
len(item_id_train.unique())

可以看到，并不是所有电影都在训练集里有评分。


In [ ]:
movies_not_train = (set(all_ratings['item_id']) -set(item_id_train))
for m in movies_not_train:
    print(m,movies_names.loc[m]['item_name'])

In [ ]:
user_id_train.iloc[:5]

In [ ]:
item_id_train.iloc[:5]

In [ ]:
rating_train.iloc[:5]

## 3. 模型

我们可以把数据集喂给 `FactorizationModel` 类——一个 sklearn 风格的对象，让我们能够训练和评估显式分解模型。

模型内部使用 `Model_dot` 类来表示用户和物品。它由 4 个 `embedding` 层组成：

- 一个 `(num_users x latent_dim)` 的 embedding 层表示用户，
- 一个 `(num_items x latent_dim)` 的 embedding 层表示物品，
- 一个 `(num_users x 1)` 的 embedding 层表示用户偏置，
- 一个 `(num_items x 1)` 的 embedding 层表示物品偏置。


In [ ]:
import torch.nn as nn
import torch

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

让我们为用户生成 [Embeddings](http://pytorch.org/docs/master/nn.html#embedding)，也就是描述用户的定长向量


In [ ]:
embedding_dim = 3
embedding_user = nn.Embedding(total_user_id, embedding_dim)
input = torch.LongTensor([[1,2,4,5],[4,3,2,0]])
embedding_user(input)

我们将使用一些自定义的 embedding 和 dataloader


In [ ]:
class ScaledEmbedding(nn.Embedding):
    """
    Embedding layer that initialises its values
    to using a normal variable scaled by the inverse
    of the embedding dimension.
    """
    def reset_parameters(self):
        """
        Initialize parameters.
        """
        self.weight.data.normal_(0, 1.0 / self.embedding_dim)
        if self.padding_idx is not None:
            self.weight.data[self.padding_idx].fill_(0)


class ZeroEmbedding(nn.Embedding):
    """
    Used for biases.
    """
    def reset_parameters(self):
        """
        Initialize parameters.
        """
        self.weight.data.zero_()
        if self.padding_idx is not None:
            self.weight.data[self.padding_idx].fill_(0)

In [ ]:
class DotModel(nn.Module):
    
    def __init__(self,
                 num_users,
                 num_items,
                 embedding_dim=32):
        
        super(DotModel, self).__init__()
        
        self.embedding_dim = embedding_dim
        
        self.user_embeddings = ScaledEmbedding(num_users, embedding_dim)
        self.item_embeddings = ScaledEmbedding(num_items, embedding_dim)
        self.user_biases = ZeroEmbedding(num_users, 1)
        self.item_biases = ZeroEmbedding(num_items, 1)
                
        
    def forward(self, user_ids, item_ids):
        
        #
        # 你的代码
        #
        return 


In [ ]:
net = DotModel(total_user_id,total_item_id).to(device)

现在在一个小 batch 上测试你的网络。


In [ ]:
batch_users_np = user_id_train.values[:5].astype(np.int32)
batch_items_np = item_id_train.values[:5].astype(np.int32)
batch_ratings_np = rating_train[:5].values.astype(np.float32)
batch_users_tensor = torch.LongTensor(batch_users_np).to(device)
batch_items_tensor = torch.LongTensor(batch_items_np).to(device)
batch_ratings_tensor = torch.tensor(batch_ratings_np).to(device)

In [ ]:
predictions = net(batch_users_tensor,batch_items_tensor)

In [ ]:
predictions

我们将使用下面定义的 MSE 损失：


In [ ]:
def regression_loss(predicted_ratings, observed_ratings):
    return ((observed_ratings - predicted_ratings) ** 2).mean()

In [ ]:
loss_fn = regression_loss
loss = loss_fn(predictions, batch_ratings_tensor)

In [ ]:
loss

通过在这个小 batch 上过拟合你的网络，确认网络在学东西（在下面的单元格里你应该达到低于 0.5 的损失）。


In [ ]:
net = DotModel(total_user_id,total_item_id).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr = 0.1)
for e in range(15):
    #
    # 你的代码
    #

In [ ]:
def shuffle(*arrays):

    random_state = np.random.RandomState()
    shuffle_indices = np.arange(len(arrays[0]))
    random_state.shuffle(shuffle_indices)

    if len(arrays) == 1:
        return arrays[0][shuffle_indices]
    else:
        return tuple(x[shuffle_indices] for x in arrays)

In [ ]:
def minibatch(batch_size, *tensors):

    if len(tensors) == 1:
        tensor = tensors[0]
        for i in range(0, len(tensor), batch_size):
            yield tensor[i:i + batch_size]
    else:
        for i in range(0, len(tensors[0]), batch_size):
            yield tuple(x[i:i + batch_size] for x in tensors)



In [ ]:
import imp
import numpy as np

import torch.optim as optim

class FactorizationModel(object):
    
    def __init__(self, embedding_dim=32, n_iter=10, batch_size=256, l2=0.0,
                 learning_rate=1e-2, device=device, net=None, num_users=None,
                 num_items=None,random_state=None):
        
        self._embedding_dim = embedding_dim
        self._n_iter = n_iter
        self._learning_rate = learning_rate
        self._batch_size = batch_size
        self._l2 = l2
        self._device = device
        self._num_users = num_users
        self._num_items = num_items
        self._net = net
        self._optimizer = None
        self._loss_func = None
        self._random_state = random_state or np.random.RandomState()
             
        
    def _initialize(self):
        if self._net is None:
            self._net = DotModel(self._num_users, self._num_items, self._embedding_dim).to(self._device)
        
        self._optimizer = optim.Adam(
                self._net.parameters(),
                lr=self._learning_rate,
                weight_decay=self._l2
            )
        
        self._loss_func = regression_loss
    
    @property
    def _initialized(self):
        return self._optimizer is not None
    
    
    def fit(self, user_ids, item_ids, ratings, verbose=True):
        
        user_ids = user_ids.astype(np.int64)
        item_ids = item_ids.astype(np.int64)
        
        if not self._initialized:
            self._initialize()
            
        for epoch_num in range(self._n_iter):
            users, items, ratingss = shuffle(user_ids,
                                            item_ids,
                                            ratings)

            user_ids_tensor = torch.from_numpy(users).to(self._device)
            item_ids_tensor = torch.from_numpy(items).to(self._device)
            ratings_tensor = torch.from_numpy(ratingss).to(self._device)
            epoch_loss = 0.0

            for (minibatch_num,
                 (batch_user,
                  batch_item,
                  batch_rating)) in enumerate(minibatch(self._batch_size,
                                                         user_ids_tensor,
                                                         item_ids_tensor,
                                                         ratings_tensor)):
                
                
                # 从这里开始填写
                predictions = 
                #
                loss = 
                epoch_loss = 
                #
                #
                # 到这里填写结束
            
            epoch_loss = epoch_loss / (minibatch_num + 1)
            
            if verbose:
                print('Epoch {}: loss_train {}'.format(epoch_num, epoch_loss))
        
            if np.isnan(epoch_loss) or epoch_loss == 0.0:
                raise ValueError('Degenerate epoch loss: {}'
                                 .format(epoch_loss))
    
    
    def test(self,user_ids, item_ids, ratings):
        self._net.train(False)
        user_ids = user_ids.astype(np.int64)
        item_ids = item_ids.astype(np.int64)
        
        user_ids_tensor = torch.from_numpy(user_ids).to(self._device)
        item_ids_tensor = torch.from_numpy(item_ids).to(self._device)
        ratings_tensor = torch.from_numpy(ratings).to(self._device)
               
        predictions = self._net(user_ids_tensor, item_ids_tensor)
        
        loss = self._loss_func(ratings_tensor, predictions)
        return loss.data.item()

    def predict(self,user_ids, item_ids):
        self._net.train(False)
        user_ids = user_ids.astype(np.int64)
        item_ids = item_ids.astype(np.int64)
        
        user_ids_tensor = torch.from_numpy(user_ids).to(self._device)
        item_ids_tensor = torch.from_numpy(item_ids).to(self._device)
               
        predictions = self._net(user_ids_tensor, item_ids_tensor)
        return predictions.data   

In [ ]:
model = FactorizationModel(embedding_dim=50,  # model = FactorizationModel(embedding_dim=50,  # 潜变量维度
                                   n_iter=5,  # n_iter=5,  # 训练的 epoch 数
                                   batch_size=1024,  # batch_size=1024,  # 小批次大小
                                   learning_rate=1e-3,
                                   l2=1e-9,  # l2=1e-9,  # L2 正则化强度
                                   num_users=total_user_id,
                                   num_items=total_item_id)

In [ ]:
user_ids_train_np = user_id_train.values.astype(np.int32)
item_ids_train_np = item_id_train.values.astype(np.int32)
ratings_train_np = rating_train.values.astype(np.float32)
user_ids_test_np = user_id_test.values.astype(np.int64)
item_ids_test_np = item_id_test.values.astype(np.int64)
ratings_test_np = rating_test.values.astype(np.float32)

In [ ]:
model.fit(user_ids_train_np, item_ids_train_np, ratings_train_np)

In [ ]:
model.test(user_ids_test_np, item_ids_test_np, ratings_test_np  )

In [ ]:
print(model._net)

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

test_preds = model.predict(user_ids_test_np, item_ids_test_np)
print("Final test RMSE: %0.3f" % np.sqrt(mean_squared_error(test_preds.cpu(), ratings_test_np)))
print("Final test MAE: %0.3f" % mean_absolute_error(test_preds.cpu(), ratings_test_np))

你可以和 [Surprise](https://github.com/NicolasHug/Surprise) 对比


## 4. 最好和最差的电影


获取电影的名字（应该有更好的方法，请提供替代方案！）


In [ ]:
list_movies_names = []
list_item_ids = []
with open(op.join(ML_100K_FOLDER, 'u.item'), encoding = "ISO-8859-1") as fp:
    for line in fp:
        list_item_ids.append(line.split('|')[0])
        list_movies_names.append(line.split('|')[1])
        
movies_names = pd.DataFrame(list(zip(list_item_ids, list_movies_names)), 
               columns =['item_id', 'item_name']) 
movies_names.head()

In [ ]:
item_bias_np = model._net.item_biases.weight.data.cpu().numpy()

In [ ]:
movies_names['biases'] = pd.Series(item_bias_np.T[0], index=movies_names.index)

In [ ]:
movies_names.head()

In [ ]:
movies_names.shape

In [ ]:
indices_item_train = np.sort(item_id_train.unique())
movies_names = movies_names.loc[indices_item_train]
movies_names.shape

In [ ]:
movies_names = movies_names.sort_values(ascending=False,by=['biases'])

最好的电影


In [ ]:
movies_names.head(10)

最差的电影


In [ ]:
movies_names.tail(10)

## 5. PCA


In [ ]:
item_emb_np = model._net.item_embeddings.weight.data.cpu().numpy()
item_emb_np.shape

In [ ]:
from sklearn.decomposition import PCA
from operator import itemgetter

pca = PCA(n_components=3)
latent_fac = pca.fit_transform(item_emb_np)

In [ ]:
movie_comp = [(f, i) for f,i in zip(latent_fac[:,1], list_movies_names)]

In [ ]:
sorted(movie_comp, key=itemgetter(0), reverse=True)[:10]

In [ ]:
sorted(movie_comp, key=itemgetter(0), reverse=False)[:10]

In [ ]:
g = all_ratings.groupby('item_name')['ratings'].count()
most_rated_movies = g.sort_values(ascending=False).index.values[:1000]
most_rated_movies[:10]

In [ ]:
idxs = range(50)
txt_movies_names = most_rated_movies[:len(idxs)]
X = latent_fac[idxs,0]
Y = latent_fac[idxs,2]
plt.figure(figsize=(15,15))
plt.scatter(X, Y)
for i, x, y in zip(txt_movies_names, X, Y):
    plt.text(x+0.01,y-0.01,i, fontsize=11)
plt.show()

## 6. SPOTLIGHT

上面写的代码是 [SPOTLIGHT](https://github.com/maciejkula/spotlight) 的简化版本


一旦你用 `conda install -c maciejkula -c pytorch spotlight=0.1.5` 安装好它，就可以对比结果了……


[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)